In [ ]:
import compneurovis as cnv


In [ ]:
DT = 0.025
DISPLAY_DT = 0.025
FLUSH_DT = 0.1
STIM_DURATION = 500.0


def build_source():
    from jax import config
    config.update("jax_enable_x64", True)

    import numpy as np
    import jaxley as jx
    from jaxley.channels import HH

    import compneurovis as cnv

    comp = jx.Compartment()
    branch = jx.Branch(comp, ncomp=1)
    xyzr = [np.array([[0.0, 0.0, 0.0, 10.0], [20.0, 0.0, 0.0, 10.0]], dtype=np.float64)]
    cell = jx.Cell([branch], parents=[-1], xyzr=xyzr)
    cell.insert(HH())
    cell.meta_name = "soma"

    def setup(network, cells):
        network.stimulate(jx.step_current(
            i_delay=0.0,
            i_dur=STIM_DURATION,
            i_amp=0.1,
            delta_t=DT,
            t_max=STIM_DURATION,
        ))

    src = cnv.jaxley.source(
        cells=[cell],
        setup=setup,
        dt=DT,
        display_dt=DISPLAY_DT,
        flush_dt=FLUSH_DT,
        v_init=-70.0,
    )
    morph = src.morphology(color_limits=(-80.0, 50.0))
    volt = src.line(
        "Selected voltage",
        source=morph.selection,
        x_unit="ms",
        rolling_window=500.0,
        max_refresh_hz=15.0,
        color="#00d2be"
    )
    cnv.layout(((morph, volt),))
    return src


In [ ]:
widget = cnv.show(build_source)
widget
